In [ ]:
import os
os.chdir(r'c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks')
print(f"Current working directory: {os.getcwd()}")

# Steps 6–9: Hybrid & Re-ranking Models (C, D, E, F)

This notebook explores four progressively refined models after diagnosing the weaknesses of the standalone BM25 and Transformer approaches:

- **Model C** — Contextual Transformer (raw text, NOT keywords) — diagnoses whether context helps
- **Model D** — Hybrid Score Fusion (70% BM25 / 30% Transformer)
- **Model E** — Two-Stage Re-ranking (BM25 retrieves top-100, Transformer re-ranks)
- **Model F** — Optimized Score Fusion (85% BM25 / 15% Transformer) — **Best performing model**

All models use the **same 50/50 train/test split** (seed=42) to ensure fair comparison with the baseline.

In [ ]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split (Same Seed as Baseline)

In [ ]:
# Load prepared data
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)

eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')
print(f"Total valid places with text and metadata: {len(df_merged)}")

# Identical split to BM25 baseline
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df  = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train (Queries): {len(train_df)} | Test (Corpus): {len(test_df)}")

### 2. Evaluation Functions (Type-Aware)

**Key design decision:** `extract_subcategories` is *type-aware*. It only compares metadata that is semantically relevant for the venue type (per assignment spec lines 34–39):
- Hotels → `priceRange`
- Restaurants → `restaurantType`, `restaurantTypeCuisine`
- Attractions (A/AP) → `activiteSubType`

In [ ]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    """Level 1 error: rank of first match on typeR (H/R/A/AP). Error = rank index."""
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    Type-aware subcategory extraction.
    Per assignment spec:
      H  (Hotel)          -> priceRange
      R  (Restaurant)     -> restaurantType, restaurantTypeCuisine
      A / AP (Attraction) -> activiteSubType
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                for p in str(val).split(','):
                    cats.add(p.strip().lower())
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    return cats

def eval_level_2(query_subcats, sorted_test_indices, test_subcats_list):
    """Level 2 error: rank of first test place sharing at least one subcategory."""
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None

# Pre-calculate test subcategories
test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]
test_df['subcategories'] = test_subcats_list
print("Evaluation functions ready.")

---
## Model C — Contextual Transformer (Raw Text)

**Hypothesis:** Keywords strip context. Maybe using the first 500 characters of raw reviews will give the Transformer richer language signals to compare against.

**Spoiler:** This is an *experimental failure* — it performs worse than keywords. The reason is that the first 500 characters of a TripAdvisor review are usually introductory fluff rather than descriptive content, making the vectors noisier.

In [ ]:
model_c = SentenceTransformer('all-MiniLM-L6-v2')

# Use first 500 chars of raw review text instead of keywords
test_texts_ctx  = [str(t)[:500] for t in test_df['review'].fillna('').tolist()]
train_texts_ctx = [str(t)[:500] for t in train_df['review'].fillna('').tolist()]

print("Encoding Test set (Contextual)...")
test_vecs_c  = model_c.encode(test_texts_ctx, show_progress_bar=True)
print("Encoding Train set (Contextual)...")
train_vecs_c = model_c.encode(train_texts_ctx, show_progress_bar=True)

In [ ]:
lvl1_errors, lvl2_errors = [], []
start_time = time.time()

sims_c = cosine_similarity(train_vecs_c, test_vecs_c)
test_type_array = test_df['typeR'].values

for i in range(len(train_df)):
    row = train_df.iloc[i]
    ranked_indices = np.argsort(sims_c[i])[::-1]
    test_typeR_list = test_type_array[ranked_indices].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None: lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices, test_subcats_list)
    if err_2 is not None: lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model C (Contextual Transformer) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model C (Contextual Transformer) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")
print("\nConclusion: Worse than keyword-based models. Raw intro text is too noisy.")

---
## Model D — Hybrid Score Fusion (70% BM25 / 30% Transformer)

**Strategy:** Combine BM25's exact keyword precision with the Transformer's semantic depth via weighted score fusion. BM25 scores are min-max normalized to [0,1] before blending.

In [ ]:
# BM25 Index
corpus_kw = test_df['top_100_words'].fillna('').astype(str).tolist()
bm25 = BM25Okapi([doc.split(' ') for doc in corpus_kw])

# Transformer (keyword-based for stability)
model_d = SentenceTransformer('all-MiniLM-L6-v2')
test_kw_d  = test_df['top_100_words'].fillna('').tolist()
train_kw_d = train_df['top_100_words'].fillna('').tolist()

# Encode with raw context (first 500 chars) matching the hybrid design in full.py
test_texts_d  = [str(t)[:500] for t in test_df['review'].fillna('').tolist()]
train_texts_d = [str(t)[:500] for t in train_df['review'].fillna('').tolist()]
test_vecs_d   = model_d.encode(test_texts_d,  show_progress_bar=False)
train_vecs_d  = model_d.encode(train_texts_d, show_progress_bar=False)
trans_sims_d  = cosine_similarity(train_vecs_d, test_vecs_d)
print("BM25 + Transformer ready.")

In [ ]:
lvl1_errors, lvl2_errors = [], []
start_time = time.time()
print("Combining scores and evaluating...")

for i in range(len(train_df)):
    row = train_df.iloc[i]
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        continue

    # BM25 scores (normalized)
    bm25_scores = bm25.get_scores(query_text.split(' '))
    bm25_norm   = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)

    # Hybrid: 70% BM25 + 30% Transformer
    hybrid_scores  = 0.7 * bm25_norm + 0.3 * trans_sims_d[i]
    ranked_indices = np.argsort(hybrid_scores)[::-1]
    test_typeR_list = test_df['typeR'].values[ranked_indices].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None: lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices, test_subcats_list)
    if err_2 is not None: lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model D (Hybrid 70/30) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model D (Hybrid 70/30) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")
print("\nLevel 2 beats BM25 baseline (4.41) — but Level 1 regressed slightly.")

---
## Model E — Two-Stage Re-ranking (BM25 → Transformer)

**Strategy (Industry Standard):**
1. **Stage 1 (Retrieval):** BM25 fetches the top-100 most relevant candidates quickly.
2. **Stage 2 (Re-ranking):** The Transformer only re-ranks those 100 candidates using semantic similarity.

This avoids running the Transformer over all 918 documents, reducing noise. However, if BM25 misses a good document in Stage 1, the Transformer never sees it.

In [ ]:
# Reuse bm25 index from Model D
# Encode with keywords for this stage (more stable than raw text per experiments)
model_e = SentenceTransformer('all-MiniLM-L6-v2')
test_vecs_e  = model_e.encode(test_kw_d,  show_progress_bar=False)
train_vecs_e = model_e.encode(train_kw_d, show_progress_bar=False)
print("Two-Stage setup ready.")

In [ ]:
TOP_K = 100
lvl1_errors, lvl2_errors = [], []
start_time = time.time()

for i in range(len(train_df)):
    row = train_df.iloc[i]
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        continue

    # STAGE 1: BM25 top-K
    bm25_scores  = bm25.get_scores(query_text.split(' '))
    top_k_idx    = np.argsort(bm25_scores)[-TOP_K:][::-1]

    # STAGE 2: Transformer re-rank
    query_vec     = train_vecs_e[i].reshape(1, -1)
    cand_vecs     = test_vecs_e[top_k_idx]
    rerank_sims   = cosine_similarity(query_vec, cand_vecs).flatten()
    reranked_top  = top_k_idx[np.argsort(rerank_sims)[::-1]]

    # Remaining docs (outside top-K) kept in BM25 order
    remaining      = [idx for idx in np.argsort(bm25_scores)[::-1] if idx not in top_k_idx]
    full_ranked    = list(reranked_top) + remaining

    test_typeR_list = test_df['typeR'].values[full_ranked].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None: lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, full_ranked, test_subcats_list)
    if err_2 is not None: lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model E (Two-Stage) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model E (Two-Stage) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")
print("\nConclusion: Worse than Score Fusion (D). BM25 Stage 1 may 'filter out' semantically good docs.")

---
## Model F — Optimized Score Fusion (85% BM25 / 15% Transformer) ★ BEST MODEL

**Strategy:** Model D (70/30) improved Level 2 but degraded Level 1. By tilting the balance further towards BM25 (85/15), we reclaim Level 1 accuracy while retaining the semantic boost for Level 2.

**Result:** Best Level 2 error (4.19) across all models tested.

In [ ]:
# Reuse bm25 index; re-encode with keywords for the Transformer signal
model_f = SentenceTransformer('all-MiniLM-L6-v2')
test_vecs_f   = model_f.encode(test_kw_d,  show_progress_bar=False)
train_vecs_f  = model_f.encode(train_kw_d, show_progress_bar=False)
trans_sims_f  = cosine_similarity(train_vecs_f, test_vecs_f)
print("Optimized Hybrid setup ready.")

In [ ]:
lvl1_errors, lvl2_errors = [], []
start_time = time.time()

for i in range(len(train_df)):
    row = train_df.iloc[i]
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        continue

    bm25_scores = bm25.get_scores(query_text.split(' '))
    bm25_norm   = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)

    # OPTIMIZED WEIGHTS: 85% BM25 + 15% Transformer
    hybrid_scores  = 0.85 * bm25_norm + 0.15 * trans_sims_f[i]
    ranked_indices = np.argsort(hybrid_scores)[::-1]
    test_typeR_list = test_df['typeR'].values[ranked_indices].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None: lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices, test_subcats_list)
    if err_2 is not None: lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"Model F (Optimized Hybrid 85/15) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model F (Optimized Hybrid 85/15) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")

---
## Summary Table

| Model | L1 Error | L2 Error | Notes |
|:------|:--------:|:--------:|:------|
| BM25 Baseline | 0.64 | 4.41 | Pure keyword matching |
| Model C — Contextual Transformer | 1.23 | 11.41 | Raw text too noisy |
| Model D — Hybrid 70/30 | 0.68 | 4.32 | L2 improved, L1 degraded |
| Model E — Two-Stage Re-ranking | 0.68 | 4.64 | BM25 Stage 1 filters too aggressively |
| **Model F — Optimized 85/15** | **0.67** | **4.19** | **Best L2; recommended model** |